In [4]:
import pandas as pd
import random as rd
from itertools import combinations
import math
import os


class TS():

    def __init__(self, Path, seed, tabu_tenure):
        self.Path = Path
        self.seed = seed
        self.tabu_tenure = tabu_tenure
        self.instance_dict = self.input_data()
        self.Initial_solution = self.get_InitialSolution()
        self.tabu_str, self.Best_solution, self.Best_objvalue = self.TSearch()

    def input_data(self):
        return pd.read_excel(
            self.Path,
            names=['Job', 'weight', 'processing_time', 'due_date'],
            index_col=0
        ).to_dict('index')

    def get_tabuestructure(self):
        tabu_dict = {}
        for swap in combinations(self.instance_dict.keys(), 2):
            tabu_dict[swap] = {
                'tabu_time': 0,
                'MoveValue': 0
            }
        return tabu_dict

    def get_InitialSolution(self):
        n_jobs = len(self.instance_dict)
        initial_solution = list(range(1, n_jobs + 1))
        rd.seed(self.seed)
        rd.shuffle(initial_solution)
        return initial_solution

    def Objfun(self, solution):
        dict = self.instance_dict
        t = 0
        objfun_value = 0

        for job in solution:
            C_i = t + dict[job]["processing_time"]
            d_i = dict[job]["due_date"]
            T_i = max(0, C_i - d_i)
            W_i = dict[job]["weight"]
            objfun_value += W_i * T_i
            t = C_i

        return objfun_value

    def SwapMove(self, solution, i, j):
        solution = solution.copy()
        i_index = solution.index(i)
        j_index = solution.index(j)

        solution[i_index], solution[j_index] = (
            solution[j_index],
            solution[i_index]
        )

        return solution

    def TSearch(self):

        tenure = self.tabu_tenure
        tabu_structure = self.get_tabuestructure()

        best_solution = self.Initial_solution.copy()
        best_objvalue = self.Objfun(best_solution)

        current_solution = self.Initial_solution.copy()
        current_objvalue = self.Objfun(current_solution)

        iter = 1
        Terminate = 0
        max_terminate = 100

        while Terminate < max_terminate:

            if iter <= 10:
                print(
                    f'Iteration {iter}: Best_objvalue: {best_objvalue}'
                )

            
            for move in tabu_structure:

                candidate_solution = self.SwapMove(
                    current_solution, move[0], move[1]
                )

                candidate_objvalue = self.Objfun(candidate_solution)

                tabu_structure[move]['MoveValue'] = candidate_objvalue

            
            while True:

                best_move = min(
                    tabu_structure,
                    key=lambda x: tabu_structure[x]['MoveValue']
                )

                MoveValue = tabu_structure[best_move]["MoveValue"]
                tabu_time = tabu_structure[best_move]["tabu_time"]

                # Not Tabu
                if tabu_time < iter:

                    current_solution = self.SwapMove(
                        current_solution,
                        best_move[0],
                        best_move[1]
                    )

                    current_objvalue = self.Objfun(current_solution)

                    if current_objvalue < best_objvalue:
                        best_solution = current_solution.copy()
                        best_objvalue = current_objvalue
                        Terminate = 0
                    else:
                        Terminate += 1

                    tabu_structure[best_move]['tabu_time'] = iter + tenure

                    iter += 1
                    break

                else:

                    
                    if MoveValue < best_objvalue:

                        current_solution = self.SwapMove(
                            current_solution,
                            best_move[0],
                            best_move[1]
                        )

                        current_objvalue = self.Objfun(current_solution)

                        best_solution = current_solution.copy()
                        best_objvalue = current_objvalue

                        Terminate = 0

                        tabu_structure[best_move]['tabu_time'] = iter + tenure

                        iter += 1
                        break

                    else:
                        tabu_structure[best_move]["MoveValue"] = float('inf')

                
                if all(
                    tabu_structure[move]["MoveValue"] == float('inf')
                    for move in tabu_structure
                ):
                    Terminate = max_terminate
                    break

        print("\nTabu search completed")

        print("\nPerformed iterations: {}".format(iter))

        print(
            "Best found Solution: {} , Objvalue: {}".format(
                best_solution, best_objvalue
            )
        )

        return tabu_structure, best_solution, best_objvalue



print("Starting Tabu search\n")

test = TS(
    Path="Instance_10.xlsx",
    seed=2012,
    tabu_tenure=3
)


Starting Tabu search

Iteration 1: Best_objvalue: 237
Iteration 2: Best_objvalue: 179
Iteration 3: Best_objvalue: 129
Iteration 4: Best_objvalue: 112
Iteration 5: Best_objvalue: 96
Iteration 6: Best_objvalue: 87
Iteration 7: Best_objvalue: 80
Iteration 8: Best_objvalue: 76
Iteration 9: Best_objvalue: 74
Iteration 10: Best_objvalue: 74

Tabu search completed

Performed iterations: 109
Best found Solution: [1, 4, 2, 6, 9, 7, 8, 5, 10, 3] , Objvalue: 74
